# M3 Loan Default Predictor — Demo

End-to-end pipeline on the UCI Credit Card Default dataset:

1. Load + clean the dataset (cached locally).
2. Train an XGBoost classifier with class-imbalance weighting.
3. Evaluate on a held-out test set + 5-fold cross-validation.
4. Explain predictions globally (SHAP summary) and locally (SHAP waterfall).

All randomness is seeded with `random_state=42`.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import matplotlib.pyplot as plt

from m3_loan_default import data, model, explain

## 1. Load the data

In [ ]:
X, y = data.load()
print(f'shape: {X.shape}')
print(f'default rate: {y.mean():.4f}')
X.head()

## 2. Train XGBoost

In [ ]:
result = model.train(X, y, run_cv=True)
print(json.dumps(result.metrics, indent=2))

## 3. Confusion matrix

In [ ]:
import numpy as np
cm = np.array(result.metrics['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['No Default', 'Default'])
ax.set_yticklabels(['No Default', 'Default'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
fig.colorbar(im, ax=ax)
plt.show()

## 4. SHAP — global feature importance

In [ ]:
shap_values = explain.compute_shap_values(result.model, result.X_test.head(1000))
explain.summary_plot(shap_values, result.X_test.head(1000))
plt.show()

In [ ]:
explain.top_features(shap_values, result.X_test.head(1000), k=10)

## 5. SHAP — local explanation for a single borrower

Waterfall plot decomposes one prediction into the contributions of each feature.

In [ ]:
explain.waterfall_plot(shap_values, row_idx=0)
plt.show()

## Notes

- Honest framing: this is a portfolio project. The UCI dataset is a benchmark, not a production credit decision system.
- No data leakage controls beyond `train_test_split` stratified on the target.
- `PAY_0` (most recent repayment status) dominates feature importance, which is consistent with credit-default literature.